#MART CHINOOK


In [0]:
%sql

-- InvoiceLineID: Identifies the invoice line 
-- InvoiceID: Identifies the invoice     
-- CustomerID: Connects to dim_customer 
-- TrackID: Connects to dim_track     
-- EmployeeID: Connects to dim_employee  
-- DateKey: Connects to dim_date
-- Quantity: Measure                     
-- UnitPrice: Measure                     
-- LineAmount: Derived measure             

SELECT *
FROM workspace.mart_chinook.fact_invoice_line;

In [0]:
%sql

-- create the dimensions

-- dim customer
-- Grain: One row = one customer
CREATE OR REPLACE TABLE workspace.mart_chinook.dim_customer AS

SELECT
    CustomerID,
    FirstName,
    LastName,
    Company,
    City,
    State,
    Country,
    Email,
    SupportRepId
FROM workspace.clean_chinook.clean_customer;

-- dim employee
-- Grain: One row = one employee
CREATE OR REPLACE TABLE workspace.mart_chinook.dim_employee AS

SELECT
    EmployeeID,
    FirstName,
    LastName,
    Title,
    ReportsTo,
    HireDate,
    City,
    State,
    Country,
    Email
FROM workspace.clean_chinook.clean_employee;

-- dim track
-- Grain: One row = one track
CREATE OR REPLACE TABLE workspace.mart_chinook.dim_track AS

SELECT
    t.TrackID,
    t.Name AS TrackName,

    a.AlbumID,
    a.Title AS AlbumTitle,

    ar.ArtistID,
    ar.Name AS ArtistName,

    g.GenreID,
    g.Name AS GenreName,

    mt.MediaTypeID,
    mt.Name AS MediaTypeName,

    t.Composer,
    t.Milliseconds,
    t.Bytes

FROM workspace.clean_chinook.clean_track t

LEFT JOIN workspace.clean_chinook.clean_album a
    ON t.AlbumID = a.AlbumID

LEFT JOIN workspace.clean_chinook.clean_artist ar
    ON a.ArtistID = ar.ArtistID

LEFT JOIN workspace.clean_chinook.clean_genre g
    ON t.GenreID = g.GenreID

LEFT JOIN workspace.clean_chinook.clean_media_type mt
    ON t.MediaTypeID = mt.MediaTypeID;

-- dim date (derivied from InvoiceDate from invoice table)
-- 
CREATE OR REPLACE TABLE workspace.mart_chinook.dim_date AS

SELECT DISTINCT

    CAST(
        DATE_FORMAT(InvoiceDate, 'yyyyMMdd')
        AS INT
    ) AS DateKey,

    CAST(InvoiceDate AS DATE) AS FullDate,

    YEAR(InvoiceDate) AS Year,

    QUARTER(InvoiceDate) AS QuarterNumber,

    CONCAT(
        'Q',
        QUARTER(InvoiceDate)
    ) AS Quarter,

    MONTH(InvoiceDate) AS MonthNumber,

    DATE_FORMAT(
        InvoiceDate,
        'MMMM'
    ) AS MonthName,

    DATE_FORMAT(
        InvoiceDate,
        'yyyy-MM'
    ) AS YearMonth

FROM workspace.clean_chinook.clean_invoice

ORDER BY FullDate;

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM workspace.clean_chinook.clean_invoice_line)
        AS clean_invoice_line_rows,

    (SELECT COUNT(*)
     FROM workspace.mart_chinook.fact_invoice_line)
        AS fact_rows;

In [0]:
%sql

SELECT
    COUNT(*) AS fact_rows,

    SUM(CASE
        WHEN c.CustomerID IS NULL THEN 1
        ELSE 0
    END) AS unmatched_customers,

    SUM(CASE
        WHEN t.TrackID IS NULL THEN 1
        ELSE 0
    END) AS unmatched_tracks,

    SUM(CASE
        WHEN e.EmployeeID IS NULL THEN 1
        ELSE 0
    END) AS unmatched_employees,

    SUM(CASE
        WHEN d.DateKey IS NULL THEN 1
        ELSE 0
    END) AS unmatched_dates

FROM workspace.mart_chinook.fact_invoice_line f

LEFT JOIN workspace.mart_chinook.dim_customer c
    ON f.CustomerID = c.CustomerID

LEFT JOIN workspace.mart_chinook.dim_track t
    ON f.TrackID = t.TrackID

LEFT JOIN workspace.mart_chinook.dim_employee e
    ON f.EmployeeID = e.EmployeeID

LEFT JOIN workspace.mart_chinook.dim_date d
    ON f.DateKey = d.DateKey;

In [0]:
%sql

-- create the fact table
-- One row in fact_invoice_line represents one invoice line, meaning one track purchased within an invoice.

CREATE OR REPLACE TABLE workspace.mart_chinook.fact_invoice_line AS

SELECT
    il.InvoiceLineID,
    il.InvoiceID,

    i.CustomerID,

    il.TrackID,

    c.SupportRepId AS EmployeeID,

    CAST(
        DATE_FORMAT(i.InvoiceDate, 'yyyyMMdd')
        AS INT
    ) AS DateKey,

    il.Quantity,

    il.UnitPrice,

    CAST(
        il.Quantity * il.UnitPrice
        AS DECIMAL(12,2)
    ) AS LineAmount

FROM workspace.clean_chinook.clean_invoice_line il

INNER JOIN workspace.clean_chinook.clean_invoice i
    ON il.InvoiceID = i.InvoiceID

INNER JOIN workspace.clean_chinook.clean_customer c
    ON i.CustomerID = c.CustomerID;

In [0]:
%sql

-- revenue reconciliation check
SELECT
    (SELECT SUM(LineAmount)
     FROM workspace.mart_chinook.fact_invoice_line)
        AS fact_revenue,

    (SELECT SUM(Total)
     FROM workspace.clean_chinook.clean_invoice)
        AS invoice_revenue,

    (SELECT SUM(LineAmount)
     FROM workspace.mart_chinook.fact_invoice_line)
    -
    (SELECT SUM(Total)
     FROM workspace.clean_chinook.clean_invoice)
        AS difference;